In [ ]:
from pathlib import Path
import sys
_root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code/notebook_runtime.py').is_file() or (p/'tools/notebook_runtime.py').is_file())
_helper=_root/'code' if (_root/'code/notebook_runtime.py').is_file() else _root/'tools'
sys.path.insert(0,str(_helper))
import notebook_runtime
notebook_runtime.configure(globals(), 'submission_package_reference_style_20260910/reproduction/render_extended_data.ipynb')


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

HERE=Path(__file__).resolve().parent
SD=HERE.parent/'supplementary_data'
OUT=HERE/'regenerated'
(OUT/'figures').mkdir(parents=True,exist_ok=True)
(OUT/'audit').mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(HERE))
from audit_panel_alignment import require_matplotlib_panel_alignment

mpl.rcParams.update({'font.family':'Arial','font.size':7.5,'axes.labelsize':7.5,'xtick.labelsize':7,'ytick.labelsize':7,'legend.fontsize':7,'pdf.fonttype':42,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':0.6,'lines.linewidth':1.3})
BLUE='#31688E';ORANGE='#BB6B35';GRAY='#73777A';GREEN='#458B76'

def canvas(titles):
    fig,axs=plt.subplots(1,2,figsize=(183/25.4,90/25.4))
    fig.subplots_adjust(left=.105,right=.97,bottom=.18,top=.74,wspace=.42)
    for i,(ax,title) in enumerate(zip(axs,titles)):
        ax.annotate(chr(97+i),xy=(0,1),xycoords='axes fraction',xytext=(-22,15),textcoords='offset points',weight='bold',fontsize=9)
        ax.set_title(title,loc='left',fontsize=8,pad=15)
        ax.tick_params(length=3,width=.6)
    return fig,axs

def yearaxis(ax):
    ax.set_xticks([2013,2016,2020,2024]);ax.set_xlabel('End year')

def save(fig,n):
    stem=OUT/'figures'/f'ExtendedData{n}'
    fig.canvas.draw()
    require_matplotlib_panel_alignment(fig,json_out=str(OUT/'audit'/f'ExtendedData{n}.alignment.json'),tolerance_pt=1.5,gutter_tolerance_pt=1.5,require_panel_labels=True,strict=True)
    fig.savefig(str(stem)+'.pdf');fig.savefig(str(stem)+'.svg')
    fig.savefig(str(stem)+'.png',dpi=300)
    fig.savefig(str(stem)+'.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig)

def main():
    g=pd.read_csv(SD/'SD6_classification_gate_summary.csv');g=g[g.end_year!='pooled'].copy();g['year']=g.end_year.astype(int)
    fig,(a,b)=canvas(['Attribution-index gate','Baseline route floor'])
    for col,label,color,ls in [('three_declines_without_index_value_share','Without index gate',GRAY,'--'),('strict_improvement_value_share','Substantive multilayer de-risking',BLUE,'-')]:
        a.plot(g.year,100*g[col],label=label,color=color,ls=ls,marker='o',ms=2.5)
    a.set_ylabel('Deconcentration value (%)');a.set_ylim(0,35)
    b.bar(g.year,100*g.route_baseline_below_threshold_value_share,color=ORANGE,width=.65)
    b.set_ylabel('Value with baseline exposure <0.025 (%)');b.set_ylim(0,35)
    for ax in [a,b]:yearaxis(ax)
    fig.legend(*a.get_legend_handles_labels(),loc='upper center',bbox_to_anchor=(.5,.98),ncol=2,frameon=False)
    save(fig,1)
    s=pd.read_csv(SD/'SD7_structural_sensitivity/annual_structure_comparison.csv')
    o=pd.read_csv(SD/'SD7_structural_sensitivity/event_overlap.csv').set_index('variant')
    variants=['reference','weights_minus_020','weights_plus_020','global_prior_only']
    labels=['Reference','Weights −0.20','Weights +0.20','Global prior only']
    cols=[BLUE,ORANGE,GREEN,GRAY]
    fig,(a,b)=canvas(['Supplier–origin mismatch','Origin-transfer event membership'])
    for v,label,c in zip(variants,labels,cols):
        q=s[s.variant==v];a.plot(q.year,q.mismatch_pct,label=label,color=c,ls='--' if v=='global_prior_only' else '-')
    a.set_ylabel('Attributed value (%)');a.set_xlabel('Year');a.set_ylim(0,100);a.set_xticks([2012,2016,2020,2024])
    pos=np.arange(4)
    b.barh(pos,o.loc[variants,'variant_events'],color='#D7DADD',height=.65,label='All events')
    b.barh(pos,o.loc[variants,'intersection'],color=BLUE,height=.32,label='Shared with reference')
    b.set_yticks(pos,labels);b.invert_yaxis();b.set_xlabel('Event count');b.set_xlim(0,210)
    fig.legend(*a.get_legend_handles_labels(),loc='upper center',bbox_to_anchor=(.5,.99),ncol=4,frameon=False)
    fig.legend(*b.get_legend_handles_labels(),loc='lower center',bbox_to_anchor=(.75,.005),ncol=2,frameon=False,fontsize=6.5)
    save(fig,2)
    d=pd.read_csv(SD/'SD6_origin_ablation_annual_comparison.csv')
    fig,(a,b)=canvas(['Three-indicator joint reduction','Risk transfer'])
    for v,label,c,ls in [('direct_partner','Direct only',GRAY,':'),('without_origin','Direct + route',ORANGE,'--'),('integrated','Integrated',BLUE,'-')]:
        q=d[d.scenario==v].sort_values('base_year')
        for ax,col in [(a,'material_joint_value_share'),(b,'risk_transfer_value_share')]:ax.plot(q.base_year,100*q[col],label=label,color=c,ls=ls,marker='o',ms=2.2)
    for ax in [a,b]:ax.set_ylim(0,80);ax.set_ylabel('Baseline value (%)');ax.set_xlabel('Baseline year');ax.set_xticks([2012,2016,2020,2024])
    fig.legend(*a.get_legend_handles_labels(),loc='upper center',bbox_to_anchor=(.5,.98),ncol=3,frameon=False)
    save(fig,3)
    c=pd.read_csv(SD/'SD8_processing_coverage_summary.csv')
    d=pd.read_csv(SD/'SD8_processing_proxy_comparison.csv')
    fig,(a,b)=canvas(['External-output evidence overlap','National concentration comparison'])
    labs=['Mineral only','Mineral\n+ stage','Mineral\n+ stage + year']
    a.barh(np.arange(3),100*c.value_share,color=[GRAY,ORANGE,BLUE],height=.55)
    a.set_yticks(np.arange(3),labs);a.invert_yaxis();a.set_xlim(0,30);a.set_xlabel('2024 fixed-cohort trade value (%)')
    metals=['Aluminium','Cobalt','Lithium','Manganese','Nickel']
    colors=[BLUE,ORANGE,GREEN,'#9A6B9E',GRAY]
    for m,color in zip(metals,colors):
        q=d[d.metal==m];b.scatter(q.observed_output_hhi,q.throughput_hhi,s=23,c=color,label=m,zorder=3,edgecolors='white',linewidths=.3)
    b.plot([0,.7],[0,.7],ls='--',color='#B6B6B6',lw=.8)
    b.set_xlim(0,.7);b.set_ylim(0,.7);b.set_xlabel('External-output HHI');b.set_ylabel('Trade-throughput HHI')
    fig.legend(*b.get_legend_handles_labels(),loc='upper center',bbox_to_anchor=(.5,.98),ncol=5,frameon=False)
    save(fig,4)
    print('Four Extended Data figures exported')

if __name__=='__main__':main()
